In [1]:
import sys, numpy, pandas, sklearn, matplotlib
print("Python:", sys.version)
print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("scikit-learn:", sklearn.__version__)
print("Matplotlib:", matplotlib.__version__)

Python: 3.12.6 (v3.12.6:a4a2d2b0d85, Sep  6 2024, 16:08:03) [Clang 13.0.0 (clang-1300.0.29.30)]
NumPy: 2.0.2
Pandas: 2.3.3
scikit-learn: 1.6.1
Matplotlib: 3.9.4


id: S24073265

In [2]:
import pandas as pd

raw = pd.read_csv("data/almaty_apartments_raw.csv")
ids = raw["listing_id"].drop_duplicates().sample(frac=0.9, random_state=3265)
df = raw[raw["listing_id"].isin(ids)].reset_index(drop=True)

print(df.shape)
print(df.dtypes)
print(df.isna().sum())
print(df.describe(include="all").T)

(1154, 11)
listing_id           object
listed_date          object
district             object
rooms                object
area_m2              object
floor                 int64
total_floors          int64
year_built          float64
building_type        object
ceiling_height_m    float64
price_kzt            object
dtype: object
listing_id            0
listed_date           0
district              0
rooms                 0
area_m2               0
floor                 0
total_floors          0
year_built           57
building_type         0
ceiling_height_m    133
price_kzt             0
dtype: int64
                   count unique         top freq         mean         std  \
listing_id          1154   1080   ALM-11147    3          NaN         NaN   
listed_date         1154    247  2026-04-18   12          NaN         NaN   
district            1154     48       Medeu  144          NaN         NaN   
rooms               1154      6           2  443          NaN         NaN   
area_

In [14]:
df[["area_m2", "price_kzt", "rooms"]].sample(10, random_state=10)

,area_m2,price_kzt,rooms
862,96.7,124040000,4
849,84.0,75680000,3
1105,50.2,26100000,1
273,41.2,"19,530,000",1
1150,114.9 m²,"105,060,000",4
987,85.1,38420000,3
349,31.1,18670000,studio
623,46.0,33390000,2
889,51.5,25020000,2
157,77.0,43580000,3


Q1:Pandas did not parce them because they have uncorrect type.Area have just number and with m^2,price writed with and without comma,rooms has studio and etc.They are have different representation of types.

In [16]:
exact_dupes = df.duplicated().sum()
print("exact copies of another row", exact_dupes)

repost_candidates = df[df.duplicated(subset="listing_id", keep=False) & ~df.duplicated()]
print("Dublicate listing_id, but different data:", repost_candidates["listing_id"].nunique())
repost_candidates.sort_values("listing_id")

exact copies of another row 31
Dublicate listing_id, but different data: 71


,listing_id,listed_date,district,rooms,area_m2,floor,total_floors,year_built,building_type,ceiling_height_m,price_kzt
7,ALM-10003,2026-05-29,Auezov,2,48.6,2,5,1978.0,panel,2.55,26120000
966,ALM-10003,2026-05-04,Auezov,2,48.6,2,5,1978.0,panel,2.55,28320000
175,ALM-10009,2026-01-19,Medeuskiy,1,33.4,2,9,1992.0,brick,3.19,39400000
939,ALM-10009,2025-12-03,Medeuskiy,1,33.4,2,9,1992.0,brick,3.19,42700000
450,ALM-10016,2026-01-29,Auezov,1,39.7,13,25,2022.0,monolith,2.80,32100000
...,...,...,...,...,...,...,...,...,...,...,...
219,ALM-11097,2026-03-05,Bostandyk,3,84.4 m²,7,16,2014.0,monolith,2.79,82 080 000 ₸
191,ALM-11124,2026-04-01,Nauryzbay,2,49.4,5,12,1964.0,panel,NaN,18680000
1094,ALM-11124,2026-03-18,Nauryzbay,2,49.4,5,12,1964.0,panel,NaN,19580000
156,ALM-11147,2026-06-25,Almaly,2,56.7,4,9,2018.0,monolith,2.86,57130000


Q2: exact copies of another row 31
Dublicate listing_id, but different data: 71

Reposts are the same appartment but diff price so we keep with the latest date because it is the same repost with updated info.

Q3: 

| Column | Expected type | Valid range |
|---|---|---|
| district | category | one of 8: Alatau, Almaly, Auezov, Bostandyk, Medeu, Nauryzbay, Turksib, Zhetysu |
| rooms | int | 1–6 |
| area_m2 | float | ~15–250 m² |
| floor | int | 1 to total_floors |
| total_floors | int | 1–30 |
| year_built | int | 1930–2026, not 0 |
| building_type | category | panel / brick / monolith |
| ceiling_height_m | float | ~2.4–3.5 m, not 0 |
| price_kzt | int | positive, roughly 10M–200M KZT |